[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Y-Robin/MachineLearningKurs/blob/main/set15-modelle-speichern-und-laden/02b_sklearn_eigener_transformer_schritt_fuer_schritt.ipynb)

**In Colab:** Zuerst die folgende Setup-Zelle ausführen. [Anleitung zum Start](https://colab.research.google.com/github/Y-Robin/MachineLearningKurs/blob/main/set1-python-grundlagen/00_start_mit_google_colab.ipynb).


In [ ]:
# Nur in Google Colab: Repository, Bibliotheken und Arbeitsordner einrichten.
import importlib.util

if (importlib.util.find_spec("google") is not None
        and importlib.util.find_spec("google.colab") is not None):
    from pathlib import Path
    import runpy
    import subprocess

    repo = Path("/content/MachineLearningKurs")
    if not repo.exists():
        subprocess.check_call(["git", "clone", "--depth", "1", "--branch", "main",
                               "https://github.com/Y-Robin/MachineLearningKurs.git", str(repo)])
    runpy.run_path(str(repo / "colab_setup.py"))["setup"](
        'set15-modelle-speichern-und-laden/02b_sklearn_eigener_transformer_schritt_fuer_schritt.ipynb', cpu=False)


# Set 15 – 02b: Eigener Transformer, Schritt für Schritt

Ein **Transformer** bereitet Daten auf. Hier ergänzt er nur zwei Spalten: jeweils Länge × Breite. Gemeint ist ein scikit-learn-Transformer, keine Transformer-Architektur aus dem Deep Learning.

`02` verbindet diese kleine Berechnung schon mit Modellpaketen, Hilfsfunktionen und einem neuen Python-Prozess. Hier gehen wir einzeln vor: **Berechnung → Transformer → Pipeline → Speichern und Laden**. Der neue Prozess kommt als optionaler Zusatz am Ende.

## 1. Vorbereitung und Daten

Führe die Zellen von oben nach unten aus. Der Pfadblock unterstützt den Start im Kurs-Hauptordner oder im Set-15-Ordner. Er sorgt nur dafür, dass Python unsere Klassendatei findet.

In [ ]:
from pathlib import Path
import sys

set_ordner = Path.cwd()
if not (set_ordner / "custom_transformer_einfach.py").exists():
    set_ordner = set_ordner / "set15-modelle-speichern-und-laden"

if not (set_ordner / "custom_transformer_einfach.py").exists():
    raise FileNotFoundError("Bitte im Kurs-Hauptordner oder im Set-15-Ordner starten.")

if str(set_ordner) not in sys.path:
    sys.path.insert(0, str(set_ordner))

In [ ]:
from sklearn.datasets import load_iris

iris = load_iris(as_frame=True)
X = iris.data
y = iris.target

X.head(3)

## 2. Erst die Berechnung ohne Klasse

Die vier Spalten enthalten Länge und Breite von Kelch- und Blütenblättern. Wir ergänzen zwei Produkte aus Länge und Breite. Das sind einfache Flächenmerkmale, keine exakt gemessenen Blattflächen.

`copy()` verhindert, dass wir die Ausgangsdaten verändern.

In [ ]:
beispiel = X.head(3).copy()
beispiel["kelch_flaeche"] = beispiel["sepal length (cm)"] * beispiel["sepal width (cm)"]
beispiel["blueten_flaeche"] = beispiel["petal length (cm)"] * beispiel["petal width (cm)"]

beispiel

## 3. Dieselbe Rechnung als Transformer

Die Pipeline erwartet für einen Vorbereitungsschritt zwei Methoden:

| Methode | Allgemeine Aufgabe | Hier |
|---|---|---|
| `fit(X, y=None)` | Eine Regel aus Trainingsdaten lernen | Nichts zu lernen: Länge × Breite steht bereits fest. |
| `transform(X)` | Die Regel auf Daten anwenden | Zwei Spalten ergänzen und die Tabelle zurückgeben. |

`self` ist das aktuelle Objekt. `return self` gibt es nach `fit` zurück, wie scikit-learn es erwartet. `y=None` erlaubt der Pipeline, Zielwerte mitzugeben; unsere Rechnung benötigt sie nicht.

Die Basisklassen `TransformerMixin` und `BaseEstimator` liefern die scikit-learn-Anbindung, etwa `fit_transform` und den Umgang mit Modellparametern.

**Warum eine separate Datei?** Beim Laden einer gespeicherten Pipeline muss Python die Klasse wiederfinden. Deshalb steht sie in `custom_transformer_einfach.py`. Die nächste Zelle zeigt den tatsächlichen Quellcode dieser Datei.

In [ ]:
from IPython.display import Code, display
from custom_transformer_einfach import FlaechenMerkmale

display(Code(filename=str(set_ordner / "custom_transformer_einfach.py"), language="python"))

Jetzt rufen wir beide Methoden einzeln auf. Die Ausgabe entspricht Abschnitt 2.

Diese einfache Klasse erwartet einen DataFrame mit den vier Iris-Spalten. Die zusätzlichen Prüfungen aus `02` sind für das Verständnis der festen Rechenregel nicht erforderlich.

In [ ]:
transformer = FlaechenMerkmale()
transformer.fit(X)
transformierte_daten = transformer.transform(X.head(3))

transformierte_daten

## 4. Transformer und Modell verbinden

**Vier Eingabespalten → sechs Merkmale → Skalierung → Klassifikation**

Bei `fit` ergänzt der Transformer die Flächen. Der Scaler lernt Mittelwerte und Streuungen, danach lernt die logistische Regression die Klassifikation. Bei `predict` laufen dieselben Vorbereitungsschritte automatisch ab, ohne erneutes Training.

Wir teilen die Daten vor dem Training auf. Insbesondere der Scaler darf nur aus Trainingsdaten lernen.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

pipeline = Pipeline([
    ("flaechen", FlaechenMerkmale()),
    ("scaler", StandardScaler()),
    ("modell", LogisticRegression(max_iter=300)),
])

pipeline.fit(X_train, y_train)
print(f"Test-Accuracy: {pipeline.score(X_test, y_test):.1%}")

Für Vorhersagen übergeben wir weiterhin die **ursprünglichen vier Spalten**. Die Pipeline ergänzt die Flächen selbst. Ob diese Merkmale die Qualität verbessern, müsste man separat vergleichen; hier geht es um den Ablauf.

In [ ]:
probe = X_test.head(3)
vorher = pipeline.predict(probe)

print("Vorhergesagte Arten:", iris.target_names[vorher])

## 5. Speichern und wieder laden

`joblib.dump` speichert die ganze trainierte Pipeline, einschließlich der gelernten Werte von Scaler und Modell. Wir verwenden einen eigenen Ausgabeordner für `02b`.

Die eigene Klasse wird über Modul- und Klassennamen referenziert. Ihr Quellcode wird nicht automatisch in die Joblib-Datei eingebettet.

In [ ]:
import joblib

ordner = set_ordner / "artefakte" / "sklearn_custom_einfach"
ordner.mkdir(parents=True, exist_ok=True)
modell_datei = ordner / "pipeline.joblib"

joblib.dump(pipeline, modell_datei)
print("Gespeichert:", modell_datei)

In [ ]:
geladen = joblib.load(modell_datei)
nachher = geladen.predict(probe)

print("Vor dem Speichern:", vorher)
print("Nach dem Laden:   ", nachher)
assert (vorher == nachher).all()
print("Die Vorhersagen stimmen überein.")

Der grundlegende Ablauf ist vollständig. Nach dem Laden rufen wir **kein `fit`** auf: Das Modell ist bereits trainiert.

Zum Weitergeben gehören `pipeline.joblib`, `custom_transformer_einfach.py` und eine Umgebung mit passenden Versionen von scikit-learn, pandas, NumPy und joblib. Die Klassendatei muss importierbar sein, zum Beispiel neben dem ausführenden Skript liegen. Neue Eingaben benötigen dieselben Spalten in derselben Reihenfolge wie beim Training.

Lade Joblib-Dateien nur aus vertrauenswürdigen Quellen, da beim Laden Python-Code ausgeführt werden kann.

## 6. Optional: Laden in einem neuen Python-Prozess

Bisher läuft alles im Notebook. Nun testen wir das Laden ohne dessen Variablen. Dazu kopieren wir die Klassendatei in den Ausgabeordner und speichern drei Testeingaben als CSV.

Das folgende kleine Skript lädt nur Modell und Eingaben und gibt Vorhersagen aus. Es enthält kein Training.

In [ ]:
import shutil

shutil.copy2(
    set_ordner / "custom_transformer_einfach.py",
    ordner / "custom_transformer_einfach.py",
)
probe.to_csv(ordner / "eingaben.csv", index=False)

skript = """import joblib
import pandas as pd

modell = joblib.load("pipeline.joblib")
eingaben = pd.read_csv("eingaben.csv")
print(modell.predict(eingaben).tolist())
"""

(ordner / "vorhersagen.py").write_text(skript, encoding="utf-8")
print(skript)

`sys.executable` verwendet denselben Python-Interpreter wie das Notebook. `cwd` setzt den Ausgabeordner als Arbeitsverzeichnis; dort liegen Modell, Klassendatei und Eingaben. `check=True` macht einen Fehler im Skript sichtbar.

Dieser Test prüft einen frischen Prozess in derselben Umgebung. Er ersetzt keinen Test auf einem anderen Rechner mit dessen Bibliotheksversionen.

In [ ]:
import json
import subprocess

prozess = subprocess.run(
    [sys.executable, "vorhersagen.py"],
    cwd=ordner,
    capture_output=True,
    text=True,
    check=True,
)

extern = json.loads(prozess.stdout)
assert extern == vorher.tolist()
print("Vorhersagen aus dem neuen Prozess:", extern)
print("Laden ohne Notebook-Zustand funktioniert.")

## Zum Nachdenken

1. Warum muss `fit` hier nichts berechnen, beim `StandardScaler` aber schon?
2. Warum erhält `predict` vier Spalten, obwohl das Modell mit sechs Merkmalen arbeitet?
3. Welche Datei fehlt, wenn du nur `pipeline.joblib` weitergibst?

**Merksatz:** `fit` lernt, `transform` verändert Daten, die Pipeline verbindet die Schritte. Zum gespeicherten Modell gehört auch der eigene Python-Code.